# ShopMind — Data Exploration
EDA notebook: understand review distribution, product catalog, and feature engineering.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Use sample data to start; replace with full dataset after download
reviews = pd.read_csv('../data/sample/sample_reviews.csv')
products = pd.read_csv('../data/sample/sample_products.csv')
print(f'Reviews: {len(reviews)} | Products: {len(products)}')
reviews.head()

## 2. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
reviews['overall'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Stars')

reviews['overall'].value_counts(normalize=True).sort_index().plot(kind='pie', ax=axes[1], autopct='%1.0f%%')
axes[1].set_title('Rating Share')
plt.tight_layout()

## 3. Review Length Analysis

In [ ]:
reviews['text_length'] = reviews['reviewText'].str.len()
reviews['word_count'] = reviews['reviewText'].str.split().str.len()

print(reviews[['text_length', 'word_count']].describe())

reviews.groupby('overall')['word_count'].mean().plot(kind='bar', color='coral')
plt.title('Average Word Count by Rating')
plt.xlabel('Star Rating')
plt.ylabel('Avg Words')
plt.tight_layout()

## 4. Fake Review Feature Engineering Preview

In [ ]:
import re
reviews['exclamations'] = reviews['reviewText'].str.count('!')
reviews['caps_words'] = reviews['reviewText'].apply(lambda t: len(re.findall(r'\b[A-Z]{2,}\b', str(t))))
reviews['rating_extremity'] = (reviews['overall'] - 3.0).abs()

# Suspicious reviews (heuristic)
reviews['suspicious'] = (
    (reviews['text_length'] < 30) |
    (reviews['rating_extremity'] > 1.8) &
    (reviews['verified_purchase'] == 0)
).astype(int)

print(f"Suspicious: {reviews['suspicious'].sum()} / {len(reviews)} ({reviews['suspicious'].mean():.1%})")
reviews[['overall', 'text_length', 'exclamations', 'rating_extremity', 'suspicious']].head(10)

## 5. Aspect Keyword Frequency

In [ ]:
aspects = {
    'battery': ['battery', 'charge', 'charging', 'drain'],
    'sound': ['sound', 'audio', 'bass', 'treble', 'volume'],
    'comfort': ['comfortable', 'comfort', 'fit', 'ear', 'pain'],
    'price': ['price', 'cost', 'value', 'worth', 'expensive'],
    'build': ['build', 'quality', 'material', 'plastic', 'sturdy']
}

aspect_counts = {}
for aspect, keywords in aspects.items():
    pattern = '|'.join(keywords)
    aspect_counts[aspect] = reviews['reviewText'].str.lower().str.contains(pattern).sum()

pd.Series(aspect_counts).sort_values(ascending=True).plot(kind='barh', color='teal')
plt.title('Aspect Keyword Frequency in Reviews')
plt.xlabel('Number of Reviews Mentioning Aspect')
plt.tight_layout()

## 6. Next Steps
- Run `scripts/download_data.py` to get full 200K review dataset
- Run `scripts/build_index.py` to build FAISS + Chroma indexes
- Run `scripts/train_models.py` to train ABSA + fake review models
- Check MLflow UI at http://localhost:5000 for training metrics